In [3]:
import pandas as pd
import random
import os
from pyecharts import options as opts
from pyecharts.charts import Graph

# ================= ⚡ 配置区 =================
TRIPLETS_FILE = 'ppmi_knowledge_triplets.csv'
OUTPUT_FILE = "ppmi_sample_dementiakg.html"

# 🎨 双层色系方案：区分“临床诊断”与“背景知识”
COLORS = {
    # --- 临床诊断层 (Clinical Data) - 暖色 ---
    "Patient":           "#e63946",  # 🔴 病人 (鲜红)
    "Diagnostic_Info":   "#f4a261",  # 🟠 诊断信息 (人口学/分组)
    "Observed_Symptom":  "#e76f51",  # 珊瑚色 - 病人表现出的症状 (连接点)

    # --- PrimeKG 知识层 (Background Knowledge) - 冷色 ---
    "KG_Anatomy":        "#4361ee",  # 🔵 神经解剖 (深蓝) - 核心背景
    "KG_Gene":           "#4cc9f0",  # 💠 基因/蛋白 (亮蓝)
    "KG_Pathway":        "#80ffdb",  # 🟢 生物通路 (浅绿)
    "Other":             "#a8a29e"   # ⚪ 其他
}

# 核心关键词 (用于辅助分类)
# 注意：症状属于“观测到的表现”，所以归为临床层连接点
CORE_SYMPTOMS = ["Tremor", "Rigidity", "Bradykinesia", "Gait", "Dysarthria", "Hypomimia", "Facies", "Speech", "Ataxia"]

# ================= 1. 节点分类逻辑 (逻辑增强) =================

def get_node_category(node_name):
    """
    将节点明确划分为 [临床层] 和 [知识层]
    """
    node_str = str(node_name)
    
    # --- 1. 临床层 ---
    if node_str.startswith("Patient:"): 
        return "Patient"
    if node_str.startswith("Concept:") or node_str.startswith("Test:"): 
        return "Diagnostic_Info"
    
    # --- 2. 知识层与连接点 ---
    if node_str.startswith("PrimeKG:"): 
        real_name = node_str.split(":", 1)[1]
        
        # [连接点] 症状：虽然来自KG，但在本图中代表病人的表型
        if any(sym in real_name for sym in CORE_SYMPTOMS):
            return "Observed_Symptom"
        
        # [知识] 基因：通常是大写字母+数字 (如 SNCA, DRD2)
        # 简单规则：长度<10 且 (全大写 或 含数字)
        is_gene = len(real_name) < 12 and (real_name.isupper() or any(c.isdigit() for c in real_name) or "receptor" in real_name.lower())
        if is_gene and "pathway" not in real_name.lower():
            return "KG_Gene"
        
        # [知识] 解剖：通常在 anatomy_protein_present 关系中作为头节点
        # 这里我们通过名字辅助判断，更准确的判断在数据加载时的 relation
        anatomy_keywords = ["Substantia", "Striatum", "Putamen", "Caudate", "Globus", "Basal", "Cortex", "Thalamus", "Cerebell", "Lobe", "Nucleus"]
        if any(k in real_name for k in anatomy_keywords):
            return "KG_Anatomy"
            
        # 剩下的归为通路/生物过程
        return "KG_Pathway"
            
    return "Other"

def print_statistics(nodes):
    """📊 打印统计信息"""
    stats = {}
    for node in nodes:
        cat = get_node_category(node)
        stats[cat] = stats.get(cat, 0) + 1
    
    print("\n" + "="*50)
    print(f"📊 图谱分层统计 (共 {len(nodes)} 个节点)")
    print("="*50)
    print("【临床诊断层 (源数据)】")
    print(f"   👤 病人 (Patient)          : {stats.get('Patient', 0)}")
    print(f"   📝 诊断记录 (Diagnostic)   : {stats.get('Diagnostic_Info', 0)}")
    print(f"   ⚠️ 观测症状 (Symptom)      : {stats.get('Observed_Symptom', 0)}")
    print("-" * 30)
    print("【PrimeKG 知识层 (推理背景)】")
    print(f"   🧠 神经解剖 (Anatomy)      : {stats.get('KG_Anatomy', 0)}")
    print(f"   🧬 基因/蛋白 (Gene)        : {stats.get('KG_Gene', 0)}")
    print(f"   ⚙️ 生物通路 (Pathway)      : {stats.get('KG_Pathway', 0)}")
    print("="*50 + "\n")

# ================= 2. 数据抽取 (修复解剖缺失bug) =================

def load_and_filter_data():
    if not os.path.exists(TRIPLETS_FILE):
        print(f"❌ 找不到文件 {TRIPLETS_FILE}")
        return None

    print(f"🚀 正在读取数据...")
    df = pd.read_csv(TRIPLETS_FILE).astype(str)
    
    # 1. 选人：选取症状最丰富的前 6 位病人
    symptom_relations = df[df['relation'] == 'has_symptom']
    top_patients = symptom_relations['head'].value_counts().head(6).index.tolist()
    if not top_patients:
        top_patients = df[df['head'].str.startswith("Patient:")].head(6)['head'].tolist()

    print(f"   -> 选取了 {len(top_patients)} 位典型病人")

    # --- 构建分层子图 ---
    
    # Layer 1: [临床] 病人 -> 症状/属性
    layer1 = df[df['head'].isin(top_patients)]
    
    # 获取图中的症状节点
    symptom_nodes = layer1[layer1['relation'] == 'has_symptom']['tail'].unique()
    
    # Layer 2: [桥接] 症状 -> 基因/疾病机制
    # (例如: Tremor --disease_protein--> Gene)
    layer2 = df[
        (df['head'].isin(symptom_nodes)) & 
        (df['tail'].str.startswith("PrimeKG:"))
    ].head(400)
    
    # 获取图中的基因节点 (作为下一层的锚点)
    current_genes = layer2['tail'].unique()
    
    # Layer 3: [知识] 基因 <-> 解剖结构 (修复点!)
    # 我们直接查找 relation 为 'anatomy_protein_present' 的行
    # 且基因在 current_genes 里
    layer3 = df[
        (df['relation'] == 'anatomy_protein_present') & 
        (df['tail'].isin(current_genes)) 
    ].head(200)
    
    # 如果 Layer 3 为空 (说明可能基因没对上)，强制找一些核心解剖结构
    if len(layer3) < 10:
        print("   ⚠️ 自动补充解剖学节点...")
        core_anat = ["PrimeKG:Substantia nigra", "PrimeKG:Putamen", "PrimeKG:Caudate nucleus"]
        layer3_supplement = df[
            (df['head'].isin(core_anat)) & 
            (df['relation'] == 'anatomy_protein_present')
        ].head(50)
        layer3 = pd.concat([layer3, layer3_supplement])

    # Layer 4: [知识] 基因 -> 通路
    layer4 = df[
        (df['head'].isin(current_genes)) & 
        (df['tail'].str.contains("pathway|process", case=False))
    ].head(100)
    
    # 合并
    subset_df = pd.concat([layer1, layer2, layer3, layer4]).drop_duplicates()
    return subset_df

# ================= 3. 渲染 =================

def render_chart(df):
    nodes_set = set(df['head']) | set(df['tail'])
    triplets = df.values.tolist()
    
    print_statistics(nodes_set)
    
    echarts_nodes = []
    
    # 定义分类
    categories = [
        {"name": "Patient",          "itemStyle": {"color": COLORS["Patient"]}},
        {"name": "Diagnostic_Info",  "itemStyle": {"color": COLORS["Diagnostic_Info"]}},
        {"name": "Observed_Symptom", "itemStyle": {"color": COLORS["Observed_Symptom"]}},
        {"name": "KG_Anatomy",       "itemStyle": {"color": COLORS["KG_Anatomy"]}},
        {"name": "KG_Gene",          "itemStyle": {"color": COLORS["KG_Gene"]}},
        {"name": "KG_Pathway",       "itemStyle": {"color": COLORS["KG_Pathway"]}},
        {"name": "Other",            "itemStyle": {"color": COLORS["Other"]}},
    ]
    
    cat_map = {c["name"]: i for i, c in enumerate(categories)}

    for node in nodes_set:
        cat_name = get_node_category(node)
        cat_idx = cat_map.get(cat_name, 6)
        
        # 节点大小设计：两头大，中间小，强调“病人”和“解剖背景”
        sizes = {
            "Patient": 40,           # 最大
            "KG_Anatomy": 25,        # 核心背景，较大
            "Observed_Symptom": 20,  # 连接点
            "KG_Gene": 15,
            "KG_Pathway": 12,
            "Diagnostic_Info": 10,
            "Other": 8
        }
        
        echarts_nodes.append({
            "name": node,
            "symbolSize": sizes.get(cat_name, 10),
            "category": cat_idx,
            "draggable": True,
            "label": {"show": False} # 默认不显示文字，悬停显示
        })

    c = (
        Graph(init_opts=opts.InitOpts(width="100%", height="950px", page_title="PPMI KG Visualization"))
        .add(
            "",
            echarts_nodes,
            links=[{"source": h, "target": t, "value": r} for h, r, t in triplets],
            categories=categories,
            layout="force",
            repulsion=300,      # 斥力大，把结构撑开
            gravity=0.1,        # 引力小
            edge_length=[50, 100],
            is_roam=True,
            is_focusnode=True,
            edge_label=opts.LabelOpts(is_show=False),
            edge_symbol=[None, "arrow"],
            edge_symbol_size=5,
            tooltip_opts=opts.TooltipOpts(formatter="{b}") # 鼠标悬停显示名字
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title="PPMI 帕金森病异构图谱 (PPMI-DementiaKG)", 
                subtitle="🔴红色区：临床诊断数据 (CSV)  |  🔵蓝色区：PrimeKG 生物知识 (Knowledge)"
            ),
            legend_opts=opts.LegendOpts(orient="vertical", pos_left="2%", pos_top="5%")
        )
    )
    
    c.render(OUTPUT_FILE)
    print(f"✅ 最终可视化文件已生成: {OUTPUT_FILE}")

if __name__ == "__main__":
    subset_df = load_and_filter_data()
    if subset_df is not None:
        render_chart(subset_df)

🚀 正在读取数据...
   -> 选取了 6 位典型病人

📊 图谱分层统计 (共 289 个节点)
【临床诊断层 (源数据)】
   👤 病人 (Patient)          : 6
   📝 诊断记录 (Diagnostic)   : 18
   ⚠️ 观测症状 (Symptom)      : 8
------------------------------
【PrimeKG 知识层 (推理背景)】
   🧠 神经解剖 (Anatomy)      : 0
   🧬 基因/蛋白 (Gene)        : 19
   ⚙️ 生物通路 (Pathway)      : 238

✅ 最终可视化文件已生成: ppmi_sample_dementiakg.html
